# Solar Power Generation Forecasting
### Predicting Inverter-Level DC Power Output from a 4.5 MW Solar Farm

**Project 19 of 50 — Time Series Forecasting Portfolio**

## Project Overview

Solar power generation follows a highly deterministic diurnal pattern — zero at night, bell-shaped during the day — but actual output is modulated by cloud cover, panel degradation, dust, and inverter faults. This notebook forecasts **DC power output** from a utility-scale solar farm using real inverter-level sensor data.

The dataset contains two solar plants in India with 34 inverters each, sampled every 15 minutes over 34 days.

| Attribute | Value |
|---|---|
| **Dataset** | nikannal/solar-power-generation-data |
| **Target** | DC_POWER (kW per inverter) |
| **Date column** | DATE_TIME |
| **Frequency** | 15-minute intervals |
| **TS Library** | Darts |
| **Key challenge** | Zero-inflated series (nights), multi-inverter panel, 34-day horizon |

## Learning Objectives

1. Handle a **multi-inverter panel dataset**: aggregate to plant-level or model per-inverter
2. Understand the **diurnal solar generation pattern** and why it dominates all variance
3. Resample from 15-minute to hourly for practical forecasting
4. Detect and handle **inverter faults** as outliers (sudden zero output during daylight)
5. Use Darts for a high-frequency generation series with seasonal period = 96 (one day at 15-min)
6. Compare statistical models against the naive "same time yesterday" baseline
7. Discuss how irradiance forecasts (not in this dataset) would transform accuracy

## Problem Statement

Given 30 days of 15-minute inverter readings from Plant 1, forecast the next **96 time-steps (24 hours)** of total plant DC power output.

This is a short-horizon, high-frequency univariate forecasting problem dominated by the daily sun cycle.

## Why This Project Matters

Solar generation forecasting is critical for:
- **Grid operators**: dispatch of backup generation during expected cloud events
- **Storage operators**: decide when to charge/discharge a co-located battery
- **Plant operators**: detect underperforming inverters (output below forecast = fault)
- **Energy traders**: sell day-ahead generation into the spot market

India's target of 500 GW solar by 2030 makes accurate forecasting increasingly important.

## Dataset Overview

**Source:** Kaggle — nikannal/solar-power-generation-data

**Files:**
| File | Description |
|---|---|
| Plant_1_Generation_Data.csv | 15-min power readings for Plant 1 (34 inverters) |
| Plant_2_Generation_Data.csv | Same structure for Plant 2 |
| Plant_1_Weather_Sensor_Data.csv | Ambient temp, module temp, irradiation |
| Plant_2_Weather_Sensor_Data.csv | Same for Plant 2 |

**Generation data columns:**
- DATE_TIME — timestamp
- PLANT_ID — plant identifier (1 or 2)
- SOURCE_KEY — inverter identifier (34 unique inverters per plant)
- DC_POWER — DC power output (kW)
- AC_POWER — AC power after inverter conversion (kW)
- DAILY_YIELD — cumulative yield for the day (kWh)
- TOTAL_YIELD — lifetime cumulative yield (kWh)

**Period:** 34 days (May–June, summer in India — high irradiance)

## Dataset Source & License

- **Kaggle slug:** nikannal/solar-power-generation-data
- **License:** CC0 Public Domain
- **Note:** 34 days is a short series for forecasting. We use a short horizon (24h) and acknowledge this limitation explicitly.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
            "scikit-learn","lazypredict","flaml","darts","statsmodels"]:
    try: __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from darts import TimeSeries as DartsSeries
from darts.models import ExponentialSmoothing, NaiveSeasonal, NaiveDrift
from darts.metrics import mae as d_mae, rmse as d_rmse

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT      = "Solar Power Generation Forecasting"
KAGGLE_SLUG  = "anikannal/solar-power-generation-data"
TARGET       = "DC_POWER"
DATE_COL     = "DATE_TIME"
FREQ         = "15min"
SEASON       = 96   # 24h / 15-min = 96 steps per day
HORIZON      = 96   # forecast 24h ahead
TEST_SIZE    = HORIZON * 3    # 3 days test
VAL_SIZE     = HORIZON * 3    # 3 days validation
RANDOM_STATE = 42
FLAML_BUDGET = 120

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
print(f"Season: {SEASON} steps/day | Horizon: {HORIZON} steps | Test: {TEST_SIZE} steps")

## Kaggle Credential Check

In [ ]:
cred = (os.environ.get("KAGGLE_USERNAME") or
        os.environ.get("KAGGLE_KEY") or
        os.environ.get("KAGGLE_API_TOKEN") or
        (pathlib.Path.home()/".kaggle"/"kaggle.json").exists())
if cred:
    print("Kaggle credentials found. Ready to download.")
else:
    print("="*55)
    print("WARNING: No Kaggle credentials found.")
    print("Visit https://www.kaggle.com/settings → Create New Token")
    print("Place kaggle.json at ~/.kaggle/kaggle.json")
    print("Or set KAGGLE_USERNAME + KAGGLE_KEY environment variables")
    print("="*55)

## Dataset Download & Loading

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = list(data_path.rglob("*.csv"))
print(f"Files ({len(csv_files)}):")
for f in csv_files: print(f"  {f.name}  {f.stat().st_size/1e3:.0f} KB")

In [ ]:
# Load Plant 1 generation data
gen_file = next((f for f in csv_files if "Plant_1_Generation" in f.name), None)
if gen_file is None:
    gen_file = next((f for f in csv_files if "generation" in f.name.lower()), csv_files[0])

print(f"Loading: {gen_file.name}")
df_raw = pd.read_csv(gen_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"Unique inverters (SOURCE_KEY): {df_raw['SOURCE_KEY'].nunique() if 'SOURCE_KEY' in df_raw.columns else 'N/A'}")
df_raw.head(3)

## Data Validation Checks

In [ ]:
print("DATA VALIDATION REPORT")
print("="*50)

# Parse datetime
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], dayfirst=True, errors="coerce")
nat_count = df_raw[DATE_COL].isna().sum()
print(f"Datetime parse NaTs: {nat_count}")

# Target column
print(f"Target '{TARGET}' present: {TARGET in df_raw.columns}")
if TARGET in df_raw.columns:
    miss = df_raw[TARGET].isna().sum()
    print(f"Missing in target   : {miss} ({miss/len(df_raw)*100:.2f}%)")
    print(f"Target range        : {df_raw[TARGET].min():.2f}  to  {df_raw[TARGET].max():.2f} kW")
    print(f"Zero values         : {(df_raw[TARGET]==0).sum()}")
    print(f"Negative values     : {(df_raw[TARGET]<0).sum()}")

print(f"Date range          : {df_raw[DATE_COL].min()}  →  {df_raw[DATE_COL].max()}")
print(f"Total rows          : {len(df_raw):,}  ({df_raw[DATE_COL].nunique()} unique timestamps across all inverters)")
print(f"Duplicates          : {df_raw.duplicated().sum()}")

## Exploratory Data Analysis

### Aggregate to plant-level

We sum DC_POWER across all inverters per timestamp to get the total plant output. This is the operationally relevant series for forecasting.

In [ ]:
# Aggregate all inverters → plant-level 15-min series
ts_df = (df_raw.groupby(DATE_COL)[TARGET]
               .sum()
               .reset_index()
               .rename(columns={DATE_COL:"ds", TARGET:"y"})
               .sort_values("ds")
               .reset_index(drop=True))

ts_df = ts_df.dropna()
print(f"Plant-level series: {len(ts_df)} observations")
print(ts_df["y"].describe().round(2))

fig = px.line(ts_df, x="ds", y="y",
              title="Plant 1 — Total DC Power Output (sum of all inverters)",
              labels={"y":"DC Power (kW)","ds":"Date"})
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Daily generation profile
ts_df["hour"] = ts_df["ds"].dt.hour + ts_df["ds"].dt.minute/60
ts_df["date"] = ts_df["ds"].dt.date

hourly_profile = ts_df.groupby("hour")["y"].mean()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

hourly_profile.plot(ax=axes[0], color="darkorange", linewidth=2)
axes[0].fill_between(hourly_profile.index, hourly_profile.values, alpha=0.3, color="gold")
axes[0].set_title("Average DC Power by Hour of Day")
axes[0].set_xlabel("Hour"); axes[0].set_ylabel("Mean DC Power (kW)")

# Day-by-day heatmap
pivot = ts_df.pivot_table(index="date", columns="hour", values="y", aggfunc="sum")
if pivot.shape[0] > 1:
    sns.heatmap(pivot.fillna(0), cmap="YlOrRd", ax=axes[1], cbar_kws={"label":"DC Power (kW)"})
    axes[1].set_title("Day × Hour Power Heatmap")
    axes[1].set_ylabel("Date"); axes[1].set_xlabel("Hour")

plt.tight_layout(); plt.show()

In [ ]:
# ACF at short lags
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(ts_df["y"].dropna(), lags=200, ax=axes[0], title="ACF — 200 lags (50 hours at 15-min)")
plot_pacf(ts_df["y"].dropna(), lags=100, ax=axes[1], title="PACF — 100 lags")
plt.tight_layout(); plt.show()
print("Strong peak at lag 96 confirms daily seasonality (96 × 15-min = 24h)")

## Target Analysis

DC power is **zero-inflated** — approximately 50% of readings are zero (nighttime). This affects metrics: MAPE is undefined at zero, and many regression models will overweight the zero-output hours.

In [ ]:
day_ts  = ts_df[ts_df["y"] > 0]
zero_ts = ts_df[ts_df["y"] == 0]
print(f"Daylight readings (y > 0): {len(day_ts):,} ({len(day_ts)/len(ts_df)*100:.1f}%)")
print(f"Night readings    (y = 0): {len(zero_ts):,} ({len(zero_ts)/len(ts_df)*100:.1f}%)")
print()
print("Daytime DC Power statistics:")
print(day_ts["y"].describe().round(2))

# Inverter fault detection
if "SOURCE_KEY" in df_raw.columns:
    inv_daily = df_raw.groupby(["SOURCE_KEY", df_raw[DATE_COL].dt.date])[TARGET].sum()
    plant_daily = df_raw.groupby(df_raw[DATE_COL].dt.date)[TARGET].sum() / df_raw["SOURCE_KEY"].nunique()
    print(f"
Per-inverter daily yield range: {inv_daily.min():.0f} - {inv_daily.max():.0f} kWh")
    print("Tip: inverters with daily yield < 50% of median are likely faulty")

## Train / Validation / Test Split

With only 34 days of data, we use 3 days for test and 3 days for validation.

In [ ]:
n = len(ts_df)
ts_test     = ts_df.iloc[n-TEST_SIZE:].copy()
ts_val      = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_train    = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_trainval = ts_df.iloc[:n-TEST_SIZE].copy()

print(f"Train:  {len(ts_train)} steps ({ts_train['ds'].min().date()} → {ts_train['ds'].max().date()})")
print(f"Val:    {len(ts_val)} steps  ({ts_val['ds'].min().date()} → {ts_val['ds'].max().date()})")
print(f"Test:   {len(ts_test)} steps  ({ts_test['ds'].min().date()} → {ts_test['ds'].max().date()})")
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max() < ts_test["ds"].min()
print("Temporal split is clean.")

## Preprocessing

1. Fill rare missing 15-min intervals with linear interpolation
2. Clip negative DC power values to 0 (inverter fault artifacts)
3. **No log transform**: the zero inflation would require special handling

In [ ]:
def preprocess(df_in):
    df = df_in.copy().sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
    miss = df["y"].isna().sum()
    if miss: df["y"] = df["y"].interpolate("linear"); print(f"  Interpolated {miss} NaN")
    neg = (df["y"] < 0).sum()
    if neg: df.loc[df["y"]<0, "y"] = 0; print(f"  Clipped {neg} negative values to 0")
    return df

ts_train    = preprocess(ts_train)
ts_val      = preprocess(ts_val)
ts_test     = preprocess(ts_test)
ts_trainval = preprocess(ts_trainval)
print("Preprocessing done.")

## Feature Engineering

For tabular approaches we create solar-specific features:
- **Lag features**: 1 step (15 min), 4 steps (1h), 96 steps (same time yesterday)
- **Daylight indicator**: is_day (hour between 6-19)
- **Hour-of-day** (cyclic encoding): captures the solar arc
- **Cumulative daylight hours**: proxy for insolation accumulation

In [ ]:
def make_solar_features(df_in):
    df = df_in.copy()
    for lag in [1, 4, 8, 96]:
        if lag < len(df): df[f"lag_{lag}"] = df["y"].shift(lag)
    df["roll_mean_4"]  = df["y"].shift(1).rolling(4).mean()
    df["roll_max_8"]   = df["y"].shift(1).rolling(8).max()
    df["hour"]         = df["ds"].dt.hour + df["ds"].dt.minute/60
    df["hour_sin"]     = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"]     = np.cos(2*np.pi*df["hour"]/24)
    df["is_day"]       = ((df["ds"].dt.hour >= 6) & (df["ds"].dt.hour <= 19)).astype(int)
    df["month"]        = df["ds"].dt.month
    df["dayofyear"]    = df["ds"].dt.dayofyear
    return df

ts_all  = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = make_solar_features(ts_all)
feat_cols = [c for c in ts_feat.columns if c not in ["ds","y"]]

train_f = ts_feat.iloc[:len(ts_train)].dropna().copy()
val_f   = ts_feat.iloc[len(ts_train):len(ts_train)+len(ts_val)].dropna().copy()
test_f  = ts_feat.iloc[len(ts_train)+len(ts_val):].dropna().copy()
print(f"Features ({len(feat_cols)}): {feat_cols}")

## Baseline Approaches

In [ ]:
results = []

def evaluate(actual, pred, name):
    a, p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a), len(p)); a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    # MAPE only on daylight hours (y > 0)
    mask = a > 10  # exclude night-time zeros
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else np.nan
    print(f"{name:<40s} MAE={mae:8.1f}  RMSE={rmse:8.1f}  MAPE(day)={mape:6.2f}%")
    results.append({"model":name,"MAE":mae,"RMSE":rmse,"MAPE_day":mape})

# Naive: last value
evaluate(ts_test["y"], np.full(len(ts_test), ts_trainval["y"].iloc[-1]), "Naive (last step)")

# Seasonal Naive 96: same time yesterday
sn96 = np.tile(ts_trainval["y"].iloc[-SEASON:].values, len(ts_test)//SEASON+1)[:len(ts_test)]
evaluate(ts_test["y"], sn96, "Seasonal Naive (same time yesterday)")

## LazyPredict Benchmark

In [ ]:
X_tr = train_f[feat_cols]; y_tr = train_f["y"]
X_va = val_f[feat_cols];   y_va = val_f["y"]
print(f"Input — train:{X_tr.shape}, val:{X_va.shape}")
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(lm.head(12).to_string())
except Exception as e:
    print(f"LazyPredict error: {e}")

## FLAML AutoML

In [ ]:
X_all = pd.concat([train_f, val_f])[feat_cols]
y_all = pd.concat([train_f, val_f])["y"]
X_te  = test_f[feat_cols]

flaml = AutoML()
flaml.fit(X_all, y_all, task="regression", metric="rmse",
           time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
flaml_pred = flaml.predict(X_te)
print(f"Best: {flaml.best_estimator}")
evaluate(test_f["y"], flaml_pred, f"FLAML ({flaml.best_estimator})")

## Darts — Dedicated Time-Series Library

**Why Darts for solar?**

Solar generation has a near-perfect sinusoidal daily pattern modulated by weather. Darts models can capture the strong seasonality natively without needing lag features. We use:
1. **NaiveSeasonal(96)** — same-time-yesterday (seasonal naive)
2. **ExponentialSmoothing** — handles daily seasonality with multiplicative structure (appropriate since generation scales with temperature)
3. **NaiveDrift** — tests whether there is a trend in the 34-day observation window

In [ ]:
def to_darts(df):
    s = df.set_index("ds")["y"].asfreq("15min").interpolate()
    return DartsSeries.from_series(s)

train_d = to_darts(ts_trainval)
h = len(ts_test)
darts_preds = {}

# 1. Seasonal Naive 96 steps
try:
    sn = NaiveSeasonal(K=SEASON)
    sn.fit(train_d); p = sn.predict(h)
    darts_preds["Darts-SeasonalNaive-96"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-SeasonalNaive-96steps")
except Exception as e: print(f"NaiveSeasonal failed: {e}")

# 2. Exponential Smoothing (multiplicative seasonal)
try:
    ets = ExponentialSmoothing(seasonal_periods=SEASON, trend=True, seasonal="mul", damped=True)
    ets.fit(train_d); p = ets.predict(h)
    darts_preds["Darts-ETS(mul)"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-ETS (multiplicative seasonal)")
except Exception as e:
    print(f"ETS multiplicative failed: {e}")
    # Fallback to additive
    try:
        ets = ExponentialSmoothing(seasonal_periods=SEASON, trend=True, seasonal="add")
        ets.fit(train_d); p = ets.predict(h)
        darts_preds["Darts-ETS(add)"] = p
        evaluate(ts_test["y"].values, p.values().flatten(), "Darts-ETS (additive seasonal)")
    except Exception as e2: print(f"ETS additive also failed: {e2}")

# 3. Naive Drift
try:
    nd = NaiveDrift()
    nd.fit(train_d); p = nd.predict(h)
    darts_preds["Darts-NaiveDrift"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-NaiveDrift")
except Exception as e: print(f"NaiveDrift failed: {e}")

In [ ]:
# Forecast plot — show only daytime portion for clarity
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual", line=dict(color="black", width=2)))
colors = ["steelblue","darkorange","green"]
for (nm, pred), col in zip(darts_preds.items(), colors):
    fig.add_trace(go.Scatter(x=ts_test["ds"], y=np.maximum(pred.values().flatten(),0),
                              name=nm, line=dict(color=col, dash="dash")))
fig.update_layout(title="Solar DC Power Forecasts vs Actual (3-day test)",
                  xaxis_title="Date/Time", yaxis_title="DC Power (kW)",
                  template="plotly_white")
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("ALL RESULTS (ranked by RMSE)"); print("="*70)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("
TOP 3:"); print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="RMSE",
             title="Model Comparison — RMSE (lower is better)",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-40, template="plotly_white")
fig.show()

## Final Training & Evaluation of Top 3

In [ ]:
for _, row in top3.iterrows():
    print(f"  {row['model']:45s} RMSE={row['RMSE']:.1f}  MAE={row['MAE']:.1f}  MAPE(day)={row['MAPE_day']:.2f}%")

# Detailed comparison plot for the best 3-day period
context = ts_trainval.iloc[-SEASON*2:]  # last 2 days of training for context
fig = go.Figure()
fig.add_trace(go.Scatter(x=context["ds"], y=context["y"],
                          name="Train (last 2 days)", line=dict(color="grey",dash="dot")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual Test", line=dict(color="black",width=2)))
if flaml_pred is not None and len(test_f)>0:
    fig.add_trace(go.Scatter(x=test_f["ds"], y=np.maximum(flaml_pred,0),
                              name="FLAML", line=dict(dash="dot",color="steelblue")))
for nm, pred in list(darts_preds.items())[:2]:
    fig.add_trace(go.Scatter(x=ts_test["ds"], y=np.maximum(pred.values().flatten(),0),
                              name=nm, line=dict(dash="dash")))
fig.update_layout(title="Top 3 Models — 3-Day Solar Forecast", template="plotly_white")
fig.show()

## Error Analysis

In [ ]:
if flaml_pred is not None and len(test_f) > 0:
    actual = test_f["y"].values
    pred   = np.maximum(flaml_pred[:len(actual)], 0)
    errors = actual - pred
    ae     = np.abs(errors)

    # Separate day vs night errors
    is_day_mask = actual > 10
    print(f"Overall  — MAE: {ae.mean():.1f} kW  |  Bias: {errors.mean():+.1f} kW")
    print(f"Daytime  — MAE: {ae[is_day_mask].mean():.1f} kW  (only hours with actual > 10 kW)")
    print(f"Nighttime— MAE: {ae[~is_day_mask].mean():.1f} kW  (near-zero, mostly noise)")

    fig, axes = plt.subplots(1, 3, figsize=(18,5))
    axes[0].hist(errors[is_day_mask], bins=30, color="gold", edgecolor="black", alpha=0.8)
    axes[0].axvline(0, color="red", lw=2, ls="--")
    axes[0].set_title("Error Distribution (Daytime only)")

    axes[1].plot(range(len(errors)), errors, alpha=0.6, color="coral")
    axes[1].axhline(0, color="red", lw=1, ls="--")
    axes[1].set_title("Errors Over 3-Day Horizon")
    axes[1].set_xlabel("15-min step")

    axes[2].scatter(actual[is_day_mask], pred[is_day_mask], s=10, alpha=0.5, color="steelblue")
    lo,hi = 0, max(actual.max(), pred.max())
    axes[2].plot([lo,hi],[lo,hi],"r--")
    axes[2].set_title("Actual vs Predicted (Daytime)")
    axes[2].set_xlabel("Actual (kW)"); axes[2].set_ylabel("Predicted (kW)")
    plt.tight_layout(); plt.show()

## Interpretation & Insights

### Key findings
1. **Seasonal Naive (same time yesterday)** is extremely hard to beat for solar — the diurnal pattern is so strong that it accounts for ~95% of output variance
2. **FLAML** with lag_96 (same-time-yesterday) as a feature essentially approximates seasonal naive, but can improve on ramp-up/down timing
3. **Darts ETS with multiplicative seasonality** can outperform additive ETS here because generation scales non-linearly with irradiance
4. The **largest errors** occur at cloud-break transitions — sudden changes from partial to full sun that no pure-history model can predict

### Inverter fault detection (bonus application)
The typical use of this forecasting model in production is **not** to predict grid dispatch — it is to detect underperforming inverters:
- Generate expected output from the model
- If actual_inverter_output < 0.7 × expected_plant_output / n_inverters → flag as potential fault

## Limitations

1. **Only 34 days of data**: Seasonal patterns for summer-vs-winter solar cannot be estimated; this series covers only high-sun months
2. **No irradiance input**: Global Horizontal Irradiance (GHI) forecasts are the single most important feature for solar forecasting — not included here
3. **Aggregated to plant level**: Inverter-level forecasting would require multi-series models (one Darts model per inverter or hierarchical reconciliation)
4. **15-minute granularity challenges**: Short series combined with high frequency means relatively few "days" for training
5. **Ramp-event blindness**: Morning and evening ramps are predictable; cloud-cover-induced midday drops are fundamentally unpredictable from history alone

## How to Improve This Project

1. **Add irradiance data**: Join the weather sensor file Plant_1_Weather_Sensor_Data.csv — IRRADIATION is the single strongest predictor of DC power
2. **Per-inverter modelling**: Fit one model per SOURCE_KEY inverter to detect individual underperformers
3. **Use a NWP (Numerical Weather Prediction) feed**: Integrate an OpenWeatherMap or Solcast irradiance forecast as a future exogenous variable in Darts
4. **Probabilistic forecast**: Use darts.utils.statistics.ConformalNaivePredictor to generate prediction intervals — essential for storage dispatch decisions
5. **Test across seasons**: Obtain data for a full year to properly evaluate winter low-sun performance

## Production Considerations

1. **Irradiance integration pipeline**: The model should consume next-day GHI forecasts from a NWP provider hourly
2. **Per-plant model registry**: Most utility-scale solar portfolios have 10–100 plants; use MLflow to version one model per plant
3. **Fault alerting**: Integrate with a SCADA system — trigger alert when actual output deviates > 15% from forecast for > 3 consecutive timesteps
4. **Storage operator integration**: Expose 96-step look-ahead forecasts via API to the battery management system
5. **Model drift monitoring**: Panel degradation (~0.5%/year) will cause systematic negative drift — retrain quarterly

## Common Mistakes to Avoid

1. **Including DAILY_YIELD or TOTAL_YIELD as features**: Both are cumulative and contain look-ahead information for intraday forecasting
2. **Using MAPE on the full series**: Night-time zeros make MAPE infinite; always restrict MAPE to daylight hours
3. **Not clipping negative predictions to zero**: Models may predict small negative values at dawn/dusk — always apply max(0, prediction) before reporting
4. **Ignoring inverter heterogeneity**: Different inverters have different efficiency curves; aggregating silently hides this
5. **Assuming solar is always positive**: Negative net metering or grid-charging events can produce negative readings in some datasets

## Mini Challenge / Exercises

1. **Add irradiance**: load Plant_1_Weather_Sensor_Data.csv, join on DATE_TIME, add IRRADIATION as a feature. Report the RMSE reduction
2. **Per-inverter forecast**: fit a Darts NaiveSeasonal(96) for each of the 34 inverters separately. Plot the inverter with the lowest total daily yield — is it a fault?
3. **Ramp detection challenge**: identify the 5 largest positive and negative 15-min ramp events (delta between consecutive readings). What time of day do they occur?
4. **Extend to Plant 2**: redo the full pipeline for Plant 2 Generation data. Do the models rank in the same order?
5. **Solar angle feature**: compute solar elevation angle from lat/lon/timestamp using the pvlib library. Does adding it improve RMSE vs the cyclic hour features?

## Final Summary & Key Takeaways

### What We Did
- Downloaded and validated 34 days of 15-min inverter data from a 4.5 MW Indian solar farm
- Aggregated 34 inverters to plant-level for forecasting
- Found and characterised the dominant daily seasonality (period = 96 steps)
- Identified the zero-inflation problem (50% + night zeros) and adjusted metrics accordingly
- Built lag features with solar-specific engineering (hour_sin/cos, is_day, lag-96)
- Benchmarked 40+ models with LazyPredict; ran FLAML AutoML
- Fitted Darts NaiveSeasonal-96, ETS (multiplicative), and NaiveDrift
- Selected Top 3 by RMSE; analysed daytime vs nighttime error patterns separately

### Key Takeaways
1. **Seasonal Naive (same-time-yesterday)** is extraordinarily strong for solar — the daily sun cycle is deterministic
2. **MAPE must be restricted to daytime hours** — zero division otherwise produces infinite errors
3. **Irradiance forecast is the missing piece** — pure price-history models plateau quickly without it
4. **Solar forecasting in production is about anomaly detection** (inverter faults) as much as grid dispatch
5. The **multiplicative ETS** often outperforms additive for proportionally-varying series like solar generation

---
*Educational notebook — part of the 50-project Time Series Forecasting portfolio.*